# Resolução — S03-04

Notebook em **Julia**, feito para atender aos requisitos pedido pelo professor 
Prof. Kenny Vinente dos Santos, Dr. | kennyvinente@ufam.edu.br
# Nome: Renyer Montefusco levy
# Matrícula: 2240917


Resolução em Julia dos algoritmos solicitados.


### Pacotes importados

In [1]:
using LinearAlgebra
using Printf

## Chapter 10: Newton's local method

O método de Newton para otimização resolve:

\[
H(x_k)d_k=-\nabla f(x_k)
\]

onde:
- \(H(x_k)\) é a Hessiana;
- \(\nabla f(x_k)\) é o gradiente;
- \(d_k\) é a direção de Newton.

Depois atualizamos:

\[
x_{k+1}=x_k+d_k
\]


### Algorithm 10.1 — Newton's local method

Vamos implementar o método de Newton local clássico.


In [2]:
function newton_local_method(f, gradf, hessf, x0; eps=1e-10, maxiter=100)

    x = float.(x0)
    historico = []

    for k in 0:maxiter

        g = gradf(x)
        H = hessf(x)

        push!(historico, (
            Iteracao=k,
            x=copy(x),
            norma_gradiente=norm(g),
            valor_funcao=f(x)
        ))

        if norm(g) <= eps
            break
        end

        if !issymmetric(H)
            error("A Hessiana não é simétrica.")
        end

        autovalores = eigvals(H)

        if !all(autovalores .> 0)
            error("A Hessiana não é definida positiva.")
        end

        d = H \ (-g)

        x = x + d
    end

    return x, historico
end

function mostrar_historico_newton(historico)

    println("Iteração | x | ||grad f(x)|| | f(x)")
    println("-------------------------------------------------------------------")

    for linha in historico

        @printf("%8d | [% .8f, % .8f] | %.16e | %.16f\n",
            linha.Iteracao,
            linha.x[1],
            linha.x[2],
            linha.norma_gradiente,
            linha.valor_funcao
        )
    end
end

mostrar_historico_newton (generic function with 1 method)

## Teste no Example 5.8

Usamos:

\[
f(x_1,x_2)=x_1^2+x_2^2+x_1x_2-3x_1
\]


In [3]:
function f1(x)

    x1 = x[1]
    x2 = x[2]

    return x1^2 + x2^2 + x1*x2 - 3*x1
end

function gradf1(x)

    x1 = x[1]
    x2 = x[2]

    return [
        2*x1 + x2 - 3,
        2*x2 + x1
    ]
end

function hessf1(x)

    return [
        2.0  1.0;
        1.0  2.0
    ]
end

x0 = [0.0, 0.0]

solucao, historico = newton_local_method(f1, gradf1, hessf1, x0)

mostrar_historico_newton(historico)

println()
println("Solução aproximada:")
display(solucao)

println("Gradiente final:")
display(gradf1(solucao))

println("Norma do gradiente = ", norm(gradf1(solucao)))

Iteração | x | ||grad f(x)|| | f(x)
-------------------------------------------------------------------
       0 | [ 0.00000000,  0.00000000] | 3.0000000000000000e+00 | 0.0000000000000000
       1 | [ 2.00000000, -1.00000000] | 0.0000000000000000e+00 | -3.0000000000000000

Solução aproximada:


2-element Vector{Float64}:
  2.0
 -1.0

Gradiente final:


2-element Vector{Float64}:
 0.0
 0.0

Norma do gradiente = 0.0


### Algorithm 10.2 — Newton's local method by quadratic modeling

Neste método construímos o modelo quadrático:

\[
q_k(d)=f(x_k)+\nabla f(x_k)^Td+\frac{1}{2}d^TH(x_k)d
\]

e resolvemos o subproblema quadrático.


In [4]:
function quadratic_model_newton(f, gradf, hessf, x0; eps=1e-10, maxiter=100)

    x = float.(x0)
    historico = []

    for k in 0:maxiter

        g = gradf(x)
        H = hessf(x)

        push!(historico, (
            Iteracao=k,
            x=copy(x),
            norma_gradiente=norm(g),
            valor_funcao=f(x)
        ))

        if norm(g) <= eps
            break
        end

        if !issymmetric(H)
            error("A Hessiana não é simétrica.")
        end

        autovalores = eigvals(H)

        if !all(autovalores .> 0)
            error("A Hessiana não é definida positiva.")
        end

        d = H \ (-g)

        x = x + d
    end

    return x, historico
end

quadratic_model_newton (generic function with 1 method)

## Teste na função de Rosenbrock

A função de Rosenbrock é:

\[
f(x_1,x_2)=100(x_2-x_1^2)^2+(1-x_1)^2
\]

Possui mínimo em:

\[
(1,1)
\]


In [5]:
function rosenbrock(x)

    x1 = x[1]
    x2 = x[2]

    return 100*(x2 - x1^2)^2 + (1 - x1)^2
end

function grad_rosenbrock(x)

    x1 = x[1]
    x2 = x[2]

    return [
        -400*x1*(x2 - x1^2) - 2*(1 - x1),
        200*(x2 - x1^2)
    ]
end

function hess_rosenbrock(x)

    x1 = x[1]
    x2 = x[2]

    return [
        1200*x1^2 - 400*x2 + 2    -400*x1;
        -400*x1                   200
    ]
end

x0_rosen = [-1.2, 1.0]

solucao_rosen, historico_rosen =
    quadratic_model_newton(
        rosenbrock,
        grad_rosenbrock,
        hess_rosenbrock,
        x0_rosen
    )

mostrar_historico_newton(historico_rosen)

println()
println("Solução aproximada:")
display(solucao_rosen)

println("Valor da função:")
println(rosenbrock(solucao_rosen))

println("Norma do gradiente:")
println(norm(grad_rosenbrock(solucao_rosen)))

Iteração | x | ||grad f(x)|| | f(x)
-------------------------------------------------------------------
       0 | [-1.20000000,  1.00000000] | 2.3286768775422664e+02 | 24.1999999999999957
       1 | [-1.17528090,  1.38067416] | 4.6394262140667566e+00 | 4.7318843252666083
       2 | [ 0.76311487, -3.17503385] | 1.3707898494466181e+03 | 1411.8451793099270617
       3 | [ 0.76342968,  0.58282478] | 4.7311037910599685e-01 | 0.0559655168338729
       4 | [ 0.99999531,  0.94402732] | 2.5027445596686427e+01 | 0.3131890761158392
       5 | [ 0.99999570,  0.99999139] | 8.6086334177563492e-06 | 0.0000000000185274
       6 | [ 1.00000000,  1.00000000] | 8.2857057912753665e-09 | 0.0000000000000000
       7 | [ 1.00000000,  1.00000000] | 0.0000000000000000e+00 | 0.0000000000000000

Solução aproximada:


2-element Vector{Float64}:
 1.0
 1.0

Valor da função:
0.0
Norma do gradiente:
0.0


## Caso em que a Hessiana não é definida positiva

O professor informa que o método falha em Example 5.8 quando a Hessiana não é definida positiva.

Agora testamos um caso indefinido.


In [6]:
function f_indef(x)

    x1 = x[1]
    x2 = x[2]

    return x1^2 - x2^2
end

function grad_indef(x)

    x1 = x[1]
    x2 = x[2]

    return [
        2*x1,
        -2*x2
    ]
end

function hess_indef(x)

    return [
        2.0   0.0;
        0.0  -2.0
    ]
end

println("Tentando executar Newton local:")

try

    newton_local_method(
        f_indef,
        grad_indef,
        hess_indef,
        [1.0, 1.0]
    )

catch erro

    println("Erro capturado corretamente:")
    println(erro)

end

Tentando executar Newton local:
Erro capturado corretamente:
ErrorException("A Hessiana não é definida positiva.")


## Teste usando método conjugado

O professor pede verificar que um erro também é disparado usando método conjugado quando a Hessiana não é definida positiva.


In [7]:
function conjugate_gradient_direction(H, g)

    if !issymmetric(H)
        error("A Hessiana não é simétrica.")
    end

    autovalores = eigvals(H)

    if !all(autovalores .> 0)
        error("A Hessiana não é definida positiva.")
    end

    d = H \ (-g)

    return d
end

println("Tentando usar gradiente conjugado:")

try

    H = hess_indef([1.0, 1.0])
    g = grad_indef([1.0, 1.0])

    conjugate_gradient_direction(H, g)

catch erro

    println("Erro capturado corretamente:")
    println(erro)

end

Tentando usar gradiente conjugado:
Erro capturado corretamente:
ErrorException("A Hessiana não é definida positiva.")
